In [1]:
from pathlib import Path
import pandas as pd


def main():
    # Buscar data/sales.csv desde la raíz o desde notebooks.
    actual = Path.cwd()
    base = next(
        (
            carpeta
            for carpeta in (actual, actual.parent)
            if (carpeta / "data" / "sales.csv").is_file()
        ),
        None,
    )

    if base is None:
        raise FileNotFoundError(
            f"No se encontró data/sales.csv desde {actual}. "
            "Abre el notebook de PRE_10_retail_sales."
        )

    sales = pd.read_csv(base / "data" / "sales.csv")

    required = [
        "OrderID", "ProductID", "Category", "ProductName",
        "Quantity", "Price", "TotalAmount", "OrderDate",
        "CustomerID", "PaymentMethod", "SalesChannel", "IsReturned",
    ]

    missing = set(required) - set(sales.columns)
    if missing:
        raise ValueError(f"Faltan columnas: {sorted(missing)}")

    if sales[required].isna().any().any():
        raise ValueError("Hay datos vacíos en las columnas requeridas.")

    sales["OrderDate"] = pd.to_datetime(
        sales["OrderDate"], errors="raise"
    )

    for column in ["Quantity", "Price", "TotalAmount", "IsReturned"]:
        sales[column] = pd.to_numeric(sales[column], errors="raise")

    if not sales["IsReturned"].isin([0, 1]).all():
        raise ValueError("IsReturned debe contener únicamente 0 o 1.")

    if sales.empty:
        raise ValueError("El archivo sales.csv no contiene registros.")

    # Se supone que IsReturned=1 representa una devolución completa.
    sales["ReturnedAmount"] = (
        sales["TotalAmount"] * sales["IsReturned"]
    )
    sales["NetSales"] = (
        sales["TotalAmount"] - sales["ReturnedAmount"]
    )

    sales["Month"] = sales["OrderDate"].dt.to_period("M").astype(str)
    sales["DayType"] = sales["OrderDate"].dt.dayofweek.map(
        lambda day: "Fin de semana" if day >= 5 else "Entre semana"
    )

    # Función común para los resúmenes.
    def summarize(columns):
        result = (
            sales.groupby(columns, as_index=False)
            .agg(
                records=("OrderID", "size"),
                orders=("OrderID", "nunique"),
                units=("Quantity", "sum"),
                gross_sales=("TotalAmount", "sum"),
                returned_amount=("ReturnedAmount", "sum"),
                net_sales=("NetSales", "sum"),
                returned_records=("IsReturned", "sum"),
            )
        )

        result["return_rate_pct"] = (
            100 * result["returned_records"] / result["records"]
        )
        result["average_order_value"] = (
            result["gross_sales"] / result["orders"]
        )

        return result

    # 1. Ventas por categoría.
    category_summary = summarize(["Category"]).sort_values(
        "net_sales", ascending=False
    )

    # 2. Ventas entre semana y en fin de semana.
    day_type_summary = summarize(["DayType"])

    # 3. Indicadores generales.
    orders = sales["OrderID"].nunique()

    kpi_summary = pd.DataFrame([{
        "records": len(sales),
        "orders": orders,
        "customers": sales["CustomerID"].nunique(),
        "products": sales["ProductID"].nunique(),
        "units": sales["Quantity"].sum(),
        "gross_sales": sales["TotalAmount"].sum(),
        "returned_amount": sales["ReturnedAmount"].sum(),
        "net_sales": sales["NetSales"].sum(),
        "returned_records": sales["IsReturned"].sum(),
        "return_rate_pct": 100 * sales["IsReturned"].mean(),
        "average_order_value": sales["TotalAmount"].sum() / orders,
    }])

    # 4. Ventas mensuales.
    monthly_sales = summarize(["Month"]).sort_values("Month")

    # 5. Ventas por método de pago.
    payment_summary = summarize(["PaymentMethod"]).sort_values(
        "net_sales", ascending=False
    )

    # 6. Segmentos: categoría + canal, ordenados por ventas netas.
    priority_segments = summarize(
        ["Category", "SalesChannel"]
    ).sort_values(
        ["net_sales", "return_rate_pct"],
        ascending=[False, True],
    ).reset_index(drop=True)

    priority_segments.insert(
        0, "priority_rank", range(1, len(priority_segments) + 1)
    )

    # 7. Devoluciones históricas por categoría y canal.
    # Es una descripción histórica, no una predicción.
    return_risk = summarize(
        ["Category", "SalesChannel"]
    ).sort_values(
        ["return_rate_pct", "records"],
        ascending=[False, False],
    )

    # 8. Top 10 clientes por ventas netas.
    top_customers = summarize(["CustomerID"]).sort_values(
        ["net_sales", "CustomerID"],
        ascending=[False, True],
    ).head(10)

    # 9. Top 10 productos por ventas netas.
    top_products = summarize(
        ["ProductID", "ProductName", "Category"]
    ).sort_values(
        ["net_sales", "ProductID"],
        ascending=[False, True],
    ).head(10)

    # Exportar los nueve archivos.
    submission = base / "submission"
    submission.mkdir(parents=True, exist_ok=True)

    outputs = {
        "category_summary.csv": category_summary,
        "day_type_summary.csv": day_type_summary,
        "kpi_summary.csv": kpi_summary,
        "monthly_sales.csv": monthly_sales,
        "payment_summary.csv": payment_summary,
        "priority_segments.csv": priority_segments,
        "return_risk.csv": return_risk,
        "top_customers.csv": top_customers,
        "top_products.csv": top_products,
    }

    for filename, table in outputs.items():
        table.to_csv(
            submission / filename,
            index=False,
            encoding="utf-8-sig",
            float_format="%.2f",
        )
        print(f"Creado: {filename}")

    print(f"\nArchivos guardados en: {submission}")


main()

Creado: category_summary.csv
Creado: day_type_summary.csv
Creado: kpi_summary.csv
Creado: monthly_sales.csv
Creado: payment_summary.csv
Creado: priority_segments.csv
Creado: return_risk.csv
Creado: top_customers.csv
Creado: top_products.csv

Archivos guardados en: c:\Users\jg702\Documents\GitHub\std-jennifer-gomez-sanchez\PRE_10_retail_sales\submission
